# Lentils foreign-object segmentation with DynUNet — training

A lentil conveyor should carry lentils and nothing else — but stones, aluminium shards, paper,
rubber and the odd fly slip through. This notebook trains the `cuvis-ai-unet` plugin's **DynUNet**
to segment those foreign objects pixel-by-pixel on the 61-band VNIR
[`Industrial_FOD_Lentils`](https://huggingface.co/datasets/cubert-gmbh/XMR_Industrial_Foreign_Object_Detection_Lentils)
dataset.

Unlike the anomaly-detection tutorials (train on normal frames only), supervised segmentation
trains on **all** frames — object frames supervise the foreground, normal frames supervise clean
background — using the dataset's published full split (808 train / 148 val / 180 test).

Training is two-phase, the cuvis-ai plugin-family convention:

1. **Phase 1 — statistics.** A `StatisticalTrainer` fits the z-score normalizer's running stats on
   full frames.
2. **Phase 2 — gradients.** DynUNet is trained against `DiceLoss + CrossEntropyLoss` on
   foreground-biased crops (nnU-Net-style oversampling of the ~0.06 % foreground).

The pipeline graph is built **in code** here (every node instantiated inline, one explicit
`pipeline.connect`), saved with `pipeline.save_to_file` (YAML + weights), and restored in the
inference notebook. This notebook shows **both** ways cuvis-ai drives that training — a
config-driven **trainrun** and the **manual** two-phase loop — behind one `TRAINING_MODE` toggle;
pick either.

It runs a **tutorial-sized** configuration (a small net, a few epochs) so it finishes quickly; the
closing section gives the champion invocation (fg-IoU ≈ 0.79) via the `train.py` CLI.

In [ ]:
# Colab bootstrap: no-op when running locally
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("In Colab Env")
    %pip install -q cuvis-ai cuvis-ai-unet cuvis-ai-augment "cuvis-ai-dataloader[cu3s,coco]"

    import torch

    if not torch.cuda.is_available():
        print(
            "WARNING: No GPU detected. Switch via Runtime > Change runtime type > GPU. "
            "DynUNet training on CPU is very slow."
        )

In [ ]:
from pathlib import Path

import numpy as np
import torch
from cuvis_ai.node.data import CU3SDataNode
from cuvis_ai.node.monitor import TensorBoardMonitorNode
from cuvis_ai.node.normalization import ZScoreNormalizer
from cuvis_ai_augment.node.compose import AugmentationCompose
from cuvis_ai_core.data.public_datasets import PublicDatasets
from cuvis_ai_core.data.splits_io import load_splits
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.training import GradientTrainer, StatisticalTrainer
from cuvis_ai_core.utils import restore_trainrun
from cuvis_ai_dataloader.data.datamodule_npz_multi import MultiNpzDataModule
from cuvis_ai_dataloader.data.npz_converter import convert_split_manifest
from cuvis_ai_schemas.pipeline import PipelineMetadata
from cuvis_ai_schemas.training import (
    CallbacksConfig,
    DataConfig,
    DataSplitConfig,
    ModelCheckpointConfig,
    OptimizerConfig,
    TrainingConfig,
    TrainRunConfig,
)
from loguru import logger
from pytorch_lightning.callbacks import ModelCheckpoint

from cuvis_ai_unet.node.dynunet import DynUNet
from cuvis_ai_unet.node.losses import CrossEntropyLoss, DiceLoss
from cuvis_ai_unet.node.seg_metrics import SegMetrics

import utils

In [ ]:
import sys

# Full lentils frames are ~263 MB (1000x1080x61 f32); the default file-descriptor sharing strategy
# pushes them through /dev/shm and exhausts it with several DataLoader workers (workers then crash).
# file_system sharing avoids the shm/fd ceiling -- set it before any loader is built.
torch.multiprocessing.set_sharing_strategy("file_system")

# Keep the notebook output readable: log at INFO and above. Nodes emit per-step progress
# (e.g. the TensorBoard monitor) at DEBUG, a firehose across hundreds of val/test steps.
logger.remove()
logger.add(sys.stderr, level="INFO")

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
logger.info(f"Using device {device}")

## 1 · Fetch + prepare the dataset

The XMR Industrial Foreign-Object-Detection (Lentils) dataset (~57 GB) lives on Hugging Face Hub:
merged `.cu3s` sessions recorded across three acquisition days, 61 spectral bands (430–910 nm) per
pixel, and pixel-level COCO masks for 7 foreign-object classes.

Segmentation uses the dataset's full split manifest `splits.csv`:

| split | frames | role |
|---|---|---|
| train | 808 | gradient training |
| val | 148 | checkpoint selection |
| test | 180 | evaluation |

Two steps, each skipped when its output already exists:

1. **Fetch** the raw dataset with `PublicDatasets.download_dataset` (skips when the folder is
   already on disk).
2. **Convert** the manifest's frames to per-frame NPZ with `convert_split_manifest`, which bakes the
   COCO polygons into a binary `mask` (+ a multi-class `class_mask`) and emits a **universe.csv**
   (`source, index, path`) and a **splits.json** (a core `DataSplitConfig` of file-index selectors)
   that `MultiNpzDataModule` reads. `SMOKE_LIMIT` caps frames per split so a first run fits in a few
   GB; set it to 0 to convert the full split (~165 GB on disk) for a champion reproduction.

> **Prerequisite:** the convert step reads `.cu3s`, so it needs `cuvis-ai-dataloader[cu3s,coco]`
> plus the matching Cuvis C++ SDK. Already have NPZ prepared (e.g. via `gen_splits.py`)? Drop them in
> `outputs/npz_local` and this step is skipped.

In [ ]:
LENTILS_DATASET_NAME = "industrial_fod_lentils"
NPZ_DIR = Path("outputs/npz_local")
SPLITS_JSON = NPZ_DIR / "splits.json"
UNIVERSE_CSV = NPZ_DIR / "universe.csv"
SMOKE_LIMIT = 12  # frames per split to convert (0 = all 1136; the full set is ~165 GB on disk)

dataset_dir = Path("/content/data") if IN_COLAB else Path("../../data")
raw_dir = dataset_dir / "XMR_Industrial_Foreign_Object_Detection_Lentils"

if SPLITS_JSON.is_file() and UNIVERSE_CSV.is_file():
    print(f"Split artifacts already present: {SPLITS_JSON}")
else:
    PublicDatasets.download_dataset(
        LENTILS_DATASET_NAME, download_path=str(dataset_dir), force=False
    )

In [ ]:
if not (SPLITS_JSON.is_file() and UNIVERSE_CSV.is_file()):
    result = convert_split_manifest(
        raw_dir / "splits.csv",
        raw_dir,
        NPZ_DIR,
        universe_csv=UNIVERSE_CSV,
        splits_json=SPLITS_JSON,
        limit=SMOKE_LIMIT,  # 0 = convert all frames
    )
    SPLITS_JSON, UNIVERSE_CSV = result.splits_json, result.universe_csv

print("Splits JSON: ", SPLITS_JSON)
print("Universe CSV:", UNIVERSE_CSV)

## 2 · How DynUNet segmentation works

- **`DynUNet`** is a MONAI-style U-Net that adapts its depth to the input. It takes the 61-band cube
  and emits per-pixel class logits (background vs. foreground). At **training** it runs on small
  crops; at **inference** it tiles a full frame with Gaussian-blended overlaps and stitches the
  logits back together, so an arbitrarily large frame is segmented without ever holding the whole
  activation map at once.
- The **normalizer sits upstream of augmentation**, so its statistics describe full frames both when
  fitted (Phase 1) and when applied. For running-stats z-score the two orders are numerically
  equivalent (an elementwise affine commutes with cropping).
- The **foreground-biased crop** (`cuvis-ai-augment`) oversamples the tiny foreground: with some
  probability it centres the crop on a random foreground pixel, otherwise it crops uniformly. It is
  a **train-only** node — at inference the augmentation block is an identity passthrough.
- **`DiceLoss + CrossEntropyLoss`** is the standard segmentation objective: Dice handles the extreme
  class imbalance, cross-entropy stabilises early training.

The graph wired below::

    DataSource ─cube─▶ Norm ─▶ Augment ─cube─▶ DynUNet ─logits─▶ DiceLoss
        └──────mask───────────▶   └────mask──────────────────────▶ CrossEntropyLoss
                                                    └─logits─▶ SegMetrics ─▶ TensorBoard

## 3 · Tutorial configuration

Edit these to customise the run.

- **`MAX_EPOCHS`**: 1 is a quick end-to-end smoke run; set 20 for a real model.
- **`PATCH`**: crop / tile side (the champion used 128).
- **`SAMPLES_PER_FRAME`**: foreground-biased crops per train frame per epoch (the champion used 4).
- **`FEATURES`**: encoder channel widths; the small default keeps the tutorial fast (champion:
  `(32, 64, 128, 256, 512)`).
- **`TRAINING_MODE`**: `"trainrun"` (config-driven, section 5a) or `"manual"` (two-phase loop
  spelled out, section 5b). Both produce the same kind of trained artifact.

In [ ]:
MAX_EPOCHS = 1                 # 20 for a real run
PATCH = 128                    # crop / tile side (champion: 128)
SAMPLES_PER_FRAME = 4          # fg-biased crops per train frame per epoch (champion: 4)
FEATURES = (16, 32, 64)        # small tutorial net; champion: (32, 64, 128, 256, 512)
BATCH = 4
NUM_WORKERS = 0                # 0 is the robust default in a notebook/interactive kernel; the
                               # train.py CLI uses several workers (behind its __main__ guard) for speed
NORM_INIT_FRAMES = 100         # z-score running stats fit on the first N train frames
TRAINING_MODE = "trainrun"     # "trainrun" (5a) or "manual" (5b)

output_dir = Path("outputs/lentils_unet_run")
output_dir.mkdir(parents=True, exist_ok=True)
pipeline_yaml_path = output_dir / "lentils_unet.yaml"
trainrun_yaml_path = output_dir / "trainrun.yaml"

print(f"Epochs:       {MAX_EPOCHS}")
print(f"Patch:        {PATCH}")
print(f"Mode:         {TRAINING_MODE}")
print(f"Splits JSON:  {SPLITS_JSON}")
print(f"Universe CSV: {UNIVERSE_CSV}")
print(f"Output dir:   {output_dir}")

## 4 · Preview the data

Before training, look at what the model will see: a couple of object frames, false-colour RGB
(nearest bands to 650/550/450 nm, display only) with the ground-truth foreground contour. The mask
is read lazily so only the frames actually shown load their full cube.

In [ ]:
import csv as _csv

import matplotlib.pyplot as plt

with open(UNIVERSE_CSV, newline="") as f:
    universe_rows = [r for r in _csv.DictReader(f) if (r.get("path") or "").strip()]


def _resolve(rel):
    p = Path(rel)
    return p if p.is_absolute() else (Path(UNIVERSE_CSV).parent / p).resolve()


shown = 0
for r in universe_rows:
    npz = _resolve(r["path"])
    with np.load(npz) as z:  # lazy: reading only "mask" never touches the ~263 MB cube
        has_fg = "mask" in z.files and bool(np.asarray(z["mask"]).any())
    if not has_fg:
        continue
    frame = utils.load_lentils_frame(npz)
    rgb = utils.false_color(frame["cube"], frame["wavelengths"])
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].imshow(rgb)
    ax[0].set_title("false-color RGB")
    ax[0].axis("off")
    ax[1].imshow(rgb)
    ax[1].contour(frame["mask"] > 0, levels=[0.5], colors="red", linewidths=1.2)
    ax[1].set_title("+ GT foreground")
    ax[1].axis("off")
    fig.suptitle(npz.name, y=1.02)
    fig.tight_layout()
    plt.show()
    shown += 1
    if shown >= 2:
        break

## 5 · Build the segmentation pipeline

Every node is instantiated inline and wired with a single `pipeline.connect`. The pipeline is saved
to YAML right away: the trainrun in section 5a references it by path, and it is the graph both
training paths train.

`SegMetrics` (foreground IoU/Dice + pixel accuracy) runs at VAL/TEST and feeds a
`TensorBoardMonitorNode`, so validation logs comparable metric curves.

In [ ]:
pipeline = CuvisPipeline("lentils_unet")

source = CU3SDataNode(name="DataSource")
norm = ZScoreNormalizer(name="Norm", use_running_stats=True, max_init_frames=NORM_INIT_FRAMES)
# Foreground-biased crop + flips, TRAIN-only (identity passthrough at inference). The crop is the
# cuvis-ai-augment RandomForegroundBiasedCrop, applied in the graph. An I/O-cheaper alternative
# crops inside the dataset -- MultiNpzDataModule(crop_size=(PATCH, PATCH)) -- see ALL-5905; if you
# enable that, drop the RandomForegroundBiasedCrop transform below to avoid cropping twice.
augment = AugmentationCompose(
    name="Augment",
    seed=0,
    extra_transform_modules=["cuvis_ai_unet.transforms"],
    transforms=[
        {"type": "RandomForegroundBiasedCrop", "size": [PATCH, PATCH], "fg_percent": 0.5, "probabilistic": True},
        {"type": "RandomHorizontalFlip", "prob": 0.5},
        {"type": "RandomVerticalFlip", "prob": 0.5},
    ],
)
net = DynUNet(
    name="DynUNet",
    mode="2d",
    in_channels=61,
    num_classes=2,
    features=list(FEATURES),
    tile_size=PATCH,
    tile_overlap=0.5,
    tile_gaussian=True,
    tile_batch=16,
)
dice = DiceLoss(name="DiceLoss", weight=1.0)
ce = CrossEntropyLoss(name="CrossEntropyLoss", weight=1.0)
seg_metrics = SegMetrics(name="SegMetrics")
tb = TensorBoardMonitorNode(output_dir=str(output_dir / "tensorboard"), run_name=pipeline.name)

pipeline.connect(
    (source.outputs.cube, norm.inputs.data),
    (norm.outputs.normalized, augment.inputs.cube),
    (source.outputs.mask, augment.inputs.mask),
    (augment.outputs.cube, net.inputs.data),
    (net.outputs.logits, dice.inputs.logits),
    (augment.outputs.mask, dice.inputs.targets),
    (net.outputs.logits, ce.inputs.logits),
    (augment.outputs.mask, ce.inputs.targets),
    (net.outputs.logits, seg_metrics.inputs.logits),
    (augment.outputs.mask, seg_metrics.inputs.targets),
    (seg_metrics.outputs.metrics, tb.inputs.metrics),
)

pipeline.save_to_file(
    str(pipeline_yaml_path),
    metadata=PipelineMetadata(
        name=pipeline.name,
        description="DynUNet foreground segmentation on lentils VNIR NPZ (z-score + Dice/CE).",
        tags=["unet", "dynunet", "segmentation", "lentils", "hyperspectral"],
        author="cuvis.ai",
    ),
)
print("Pipeline saved:", pipeline_yaml_path)

In [ ]:
pipeline

## 5a · Train — via a trainrun (config-driven)

cuvis-ai exposes training through a `TrainRunConfig` rather than a hand-assembled Lightning trainer.
It bundles everything needed to reproduce a run: the pipeline (referenced by the YAML saved above),
the data module, the schedule, and the nodes that provide the losses and metrics. `restore_trainrun`
then runs the whole workflow — statistical init, gradient training (DynUNet unfrozen, the rest
frozen), best-checkpoint selection on `val_loss`, serialization, and val/test passes.

The saved YAML is a reproducible run spec, runnable from the terminal:

```bash
uv run restore-trainrun --trainrun-path outputs/lentils_unet_run/trainrun.yaml --mode train \
    --plugins-dir . --plugins-dir examples/lentils
```

In [ ]:
if TRAINING_MODE == "trainrun":
    trainrun = TrainRunConfig(
        name="lentils_unet",
        pipeline=pipeline_yaml_path.name,  # resolved relative to the trainrun yaml
        data=DataConfig(
            data_module="npz_multi",
            batch_size=BATCH,
            num_workers=NUM_WORKERS,
            splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
            params={
                "universe_csv": str(UNIVERSE_CSV.resolve()),
                "samples_per_frame": SAMPLES_PER_FRAME,
            },
        ),
        training=TrainingConfig(
            seed=42,
            max_epochs=MAX_EPOCHS,
            accelerator="auto",
            devices=1,
            default_root_dir=str(output_dir),
            precision="32-true",
            enable_progress_bar=True,
            enable_checkpointing=True,
            log_every_n_steps=10,
            check_val_every_n_epoch=1,
            gradient_clip_val=1.0,  # guards against a BatchNorm eval-mode divergence
            optimizer=OptimizerConfig(name="adam", lr=1e-3),
            callbacks=CallbacksConfig(
                checkpoint=ModelCheckpointConfig(
                    dirpath=str(output_dir / "checkpoints"),
                    monitor="val_loss",
                    mode="min",
                    save_top_k=1,
                    save_last=True,
                    filename="{epoch:02d}",
                )
            ),
        ),
        loss_nodes=["DiceLoss", "CrossEntropyLoss"],
        metric_nodes=["SegMetrics"],
        unfreeze_nodes=["DynUNet"],
        output_dir=str(output_dir),
    )
    trainrun.save_to_file(trainrun_yaml_path)
    print("Trainrun saved:", trainrun_yaml_path)

    # plugins_dirs: the manifest dirs restore resolves the graph + data module from -- the unet
    # plugin (repo root plugins.yaml), and augment + cuvis_ai_dataloader (examples/lentils). The
    # npz_multi data module lives in cuvis_ai_dataloader.yaml there.
    restore_trainrun(
        trainrun_yaml_path,
        mode="train",
        plugins_dirs=[str(utils.REPO_ROOT), str(utils.REPO_ROOT / "examples" / "lentils")],
    )
else:
    print(f"TRAINING_MODE={TRAINING_MODE!r} -> skipping the trainrun path (see 5b).")

## 5b · Train — manually (two-phase, spelled out)

The same training, assembled by hand: build the data module, fit the normalizer statistics
(`StatisticalTrainer`), unfreeze DynUNet, then gradient-train against Dice + CE (`GradientTrainer`).
A best-`val_loss` `ModelCheckpoint` is kept **in the callbacks list** so the trainer honours it — a
divergence that sends `val_loss` to NaN never beats the best, so `validate(ckpt_path="best")` reloads
the best pre-divergence epoch before the artifact is saved.

In [ ]:
if TRAINING_MODE == "manual":
    datamodule = MultiNpzDataModule(
        universe_csv=str(UNIVERSE_CSV),
        splits=load_splits(str(SPLITS_JSON)),
        batch_size=BATCH,
        num_workers=NUM_WORKERS,
        persistent_workers=NUM_WORKERS > 0,
        samples_per_frame=SAMPLES_PER_FRAME,
    )

    # Phase 1 -- statistics: fit the z-score running stats on full frames (Augment passes through
    # outside TRAIN, so the stats describe whole frames).
    pipeline.to(device)
    StatisticalTrainer(pipeline=pipeline, datamodule=datamodule).fit()

    # Phase 2 -- gradients: train DynUNet against Dice + CE, best-val checkpointing.
    pipeline.unfreeze_nodes_by_name(["DynUNet"])
    checkpoint = ModelCheckpoint(
        dirpath=str(output_dir / "checkpoints"),
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        save_last=True,
        verbose=True,
    )
    trainer = GradientTrainer(
        pipeline=pipeline,
        datamodule=datamodule,
        loss_nodes=[dice, ce],
        metric_nodes=[seg_metrics],
        monitors=[tb],
        training_config=TrainingConfig(
            max_epochs=MAX_EPOCHS,
            accelerator="auto",
            devices=1,
            enable_progress_bar=True,
            log_every_n_steps=1,
            enable_checkpointing=True,
            gradient_clip_val=1.0,
            check_val_every_n_epoch=1,
            optimizer=OptimizerConfig(name="adam", lr=1e-3),
        ),
        callbacks=[checkpoint],
    )
    trainer.fit()
    trainer.validate(ckpt_path="best")  # reload the best-val_loss weights into the pipeline

    manual_dir = output_dir / "trained_models"
    manual_dir.mkdir(parents=True, exist_ok=True)
    manual_artifact = manual_dir / "lentils_unet_manual.yaml"
    pipeline.save_to_file(
        str(manual_artifact),
        save_weights=True,
        metadata=PipelineMetadata(
            name=pipeline.name,
            description="Two-phase-trained lentils segmentation (best val_loss checkpoint).",
        ),
    )
    print("Manual artifact saved:", manual_artifact)
else:
    print(f"TRAINING_MODE={TRAINING_MODE!r} -> skipping the manual path (trainrun ran in 5a).")

## 6 · What the run produced

Both paths write the trained pipeline (YAML + co-located `.pt`) under `outputs/lentils_unet_run/`.
The trainrun saves it under `trained_models/lentils_unet_restored.*`; the manual path under
`trained_models/lentils_unet_manual.*`. Point the inference notebook's `PIPELINE_DIR` at that
`trained_models` dir.

In [ ]:
trained_dir = output_dir / "trained_models"
expected = (
    trained_dir / "lentils_unet_restored.yaml"
    if TRAINING_MODE == "trainrun"
    else trained_dir / "lentils_unet_manual.yaml"
)
for artifact in sorted(trained_dir.glob("*")):
    print(f"{artifact}  ({artifact.stat().st_size / 1e6:.1f} MB)")

assert expected.is_file(), f"expected trained pipeline at {expected}"
print("\nTrained pipeline:", expected)
print("Inference notebook -> set PIPELINE_DIR =", trained_dir)

## 7 · Reproducing the champion

The published numbers (fg-IoU **0.79** / fg-Dice **0.88** / image-AUROC **0.998**) come from the
full split with the deep topology `(32, 64, 128, 256, 512)` and 20 epochs — the same engine, driven
by the `train.py` CLI:

```bash
python examples/lentils/train.py \
    --universe-csv outputs/npz_local/universe.csv \
    --splits-json  outputs/npz_local/splits.json \
    --mode 2d --patch 128 --samples-per-frame 4 \
    --epochs 20 --batch 8 --num-workers 4 --val-every 1 --save-best-val --grad-clip 1.0 \
    --tensorboard --out runs/2d128
python examples/lentils/evaluate.py --pipeline runs/2d128/pipeline.yaml \
    --universe-csv outputs/npz_local/universe.csv --splits-json outputs/npz_local/splits.json
```

Continue with [`02_inference.ipynb`](02_inference.ipynb) to evaluate and visualize the artifact this
notebook just saved.